In [1]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec
from tqdm.auto import tqdm
from api_utils import Utils

# Dataset

In [2]:
ds = load_dataset('sentence-transformers/quora-duplicates', 'pair-class')
ds

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label'],
        num_rows: 404290
    })
})

In [3]:
ds['train'][20]

{'sentence1': 'Why do rockets look white?',
 'sentence2': 'Why are rockets and boosters painted white?',
 'label': 1}

In [4]:
SAMPLE_SIZE = 10000
ds_shuffled = ds['train'].shuffle().select(range(SAMPLE_SIZE))
ds_shuffled

Dataset({
    features: ['sentence1', 'sentence2', 'label'],
    num_rows: 10000
})

In [5]:
questions = []
for pair in ds_shuffled:
    questions.extend([pair['sentence1'], pair['sentence2']])
questions = list(set(questions))

print('\n'.join(questions[:10]))
print('~'*100)
print(f'Number of questions: {len(questions)}')

How do I learn machine learning?
Is the hiring process is different for the post of hr executive in consultancy, private co. or MNC? What is the procedure?
What are some little known facts about McDonald's?
How can I contact Keanu Reeves or his agent?
How can you teach yourself to take from others when you're the one who does all the giving but doesn't want to take?
Why did M.S.Dhoni left captaincy from ODI & T20?
What is the meaning of below sentence?
Which is the most addictive TV Series?
Can borderline personality disorder be cured?
Can I have my own website and use the Amazon FBA service?
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Number of questions: 19395


# Model for Sequence Embedding

In [6]:
model = SentenceTransformer('all-MiniLM-L6-v2')

In [7]:
SENTENCE_DIM = model.get_sentence_embedding_dimension()
SENTENCE_DIM

384

In [8]:
query = 'which city is the most populated in the world?'
x_emb = model.encode(query)
x_emb.shape

(384,)

# Vector Database to Store Embeddings

In [9]:
def create_index(index_name, dimension, metric="cosine"):
    # Connect
    pc = Pinecone(api_key=Utils.get_pinecone_api_key())

    # List existing indexes
    existing_indexes = [idx["name"] for idx in pc.list_indexes()]

    # Create new one
    if index_name in existing_indexes:
        print(f"Index '{index_name}' already exists → deleting ...")
        pc.delete_index(name=index_name)
        print(f"Deleted '{index_name}' successfully!")

    print(f"Creating index '{index_name}' ...")
    pc.create_index(
        name=index_name,
        dimension=dimension,
        metric=metric,
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print(f"Index '{index_name}' created successfully!")

    return pc.Index(index_name)  # returns object

In [10]:
index = create_index('first-index', dimension=SENTENCE_DIM)
index

Index 'first-index' already exists → deleting ...
Deleted 'first-index' successfully!
Creating index 'first-index' ...
Index 'first-index' created successfully!


# Upsert to DB

In [11]:
def upsert_questions(questions, index, model, batch_size=64):
    for i in tqdm(range(0, len(questions), batch_size)):
        i_end = min(i + batch_size, len(questions))

        # Create id
        ids = [str(x) for x in range(i, i_end)]
        # Create metadata (raw text, ...)
        raw_texts = [{'text': q} for q in questions[i:i_end]]
        # Create data (vector)
        vectors = model.encode(questions[i:i_end])

        # Upsert to DB (Insert + Update if similar id)
        records = zip(ids, vectors, raw_texts)
        index.upsert(vectors=records)

In [12]:
upsert_questions(questions, index, model, batch_size=512)

  0%|          | 0/38 [00:00<?, ?it/s]

# Semantic Search

In [13]:
def search_similar_questions(question, index, model, k=5):
    vector = model.encode(question).tolist()
    results = index.query(vector=vector, top_k=k, include_metadata=True, include_values=False)

    for match in results['matches']:
        score = match['score']
        text = match['metadata']['text']
        print(f"{score:.02f}: {text}")

In [14]:
question = 'how do i make chocolate cake?'
search_similar_questions(question, index, model)

0.84: How does one make a cake?
0.79: How can I make a Chocolate at home?
0.79: How do you make chocolates at home?
0.64: How do you bake a 10" cake?
0.60: How can you make crepes with pancake mix?


In [15]:
question = 'which city has the highest population in the world?'
search_similar_questions(question, index, model)

0.71: What are the world's most advanced cities?
0.63: What are the world's most technologically advanced cities?
0.62: How many cities are on Earth?
0.61: What's the most livable city in China?
0.58: Is it true that Dubai is the most expensive city in the world?
